### **COMPONENT 1: DATA EXTRACTION**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#!pip install -q requests pandas beautifulsoup4 lxml

In [3]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import time
from pathlib import Path
import json
from datetime import datetime
import numpy as np
from collections import defaultdict

In [4]:
base_dir = Path('/content/drive/MyDrive/Colab Notebooks/13F_AI_Agent')
data_dir = base_dir / 'data'
raw_dir = base_dir / 'raw_filings'

base_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)
raw_dir.mkdir(exist_ok=True)

In [5]:
HEADERS = {
    'User-Agent': 'First-Name Last-Name Email'  # REPLACE WITH YOUR INFO- required by SEC for downloading data
}

#### Find the 13F filing URL for a given CIK and period date

In [6]:
def get_filing_url(cik, period_date):
    cik = str(cik).zfill(10)

    # Use SEC submissions API to get structured data
    submissions_url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    print(f"Searching for 13F filing: CIK {cik}, Reporting Period {period_date}")

    try:
        response = requests.get(submissions_url, headers=HEADERS)
        response.raise_for_status()

        data = response.json()
        filings = data.get('filings', {}).get('recent', {})

        # Find 13F-HR filings
        for i in range(len(filings.get('form', []))):
            if filings['form'][i] == '13F-HR':
                filing_date = filings['filingDate'][i]
                accession = filings['accessionNumber'][i]
                report_date = filings.get('reportDate', [None] * (i+1))[i]

                print(f"  - Filing: {filing_date}, Reporting: {report_date}")

                if report_date == period_date:
                    # Build documents URL from accession number
                    accession_dashed = '-'.join([accession[:10], accession[10:12], accession[12:]])
                    documents_url = f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}/{accession}/{accession_dashed}-index.htm"

                    print(f"✅ Found matching filing! Filed on {filing_date}, reporting period {report_date}")
                    return accession, documents_url

        print("No 13F-HR filing found for this reporting period")
        return None, None

    except Exception as e:
        print(f"Error searching for filing: {e}")
        return None, None

In [7]:
def get_html_table_url(documents_url):
    """
    Find the HTML information table URL
    Much simpler than XML - just look for Type = "INFORMATION TABLE"

    Args:
        documents_url: URL to the filing documents page

    Returns:
        URL to the HTML information table
    """
    try:
        response = requests.get(documents_url, headers=HEADERS)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')
        table = soup.find('table', {'class': 'tableFile'})

        if not table:
            print("❌ No documents table found")
            return None

        rows = table.find_all('tr')[1:]  # Skip header

        print("   Searching for INFORMATION TABLE...")

        # Look for Type = "INFORMATION TABLE"
        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 4:
                doc_type = cols[3].text.strip().upper()

                if 'INFORMATION TABLE' in doc_type:
                    filename = cols[2].text.strip()

                    # Get the HTML file
                    if filename.lower().endswith('.html'):
                        link = cols[2].find('a')
                        if link:
                            html_url = 'https://www.sec.gov' + link['href']
                            print(f"   ✅ Found: {filename}")
                            return html_url

        print("   ❌ HTML information table not found")
        return None

    except Exception as e:
        print(f"❌ Error finding HTML file: {e}")
        return None


In [8]:
def parse_13f_html(html_url):
    """
    Parse the 13F HTML table and extract holdings
    """
    print(f"Downloading HTML table...")

    try:
        response = requests.get(html_url, headers=HEADERS)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        # DEBUG: Save the raw HTML to see what we're working with
        with open('/content/debug_html.html', 'w', encoding='utf-8') as f:
            f.write(str(soup))
        print(" Saved raw HTML to debug_html.html for inspection")

        # Find ALL tables and examine each one
        all_tables = soup.find_all('table')
        print(f"Found {len(all_tables)} total tables in HTML")

        # Examine each table to find the right one
        for table_idx, table in enumerate(all_tables):
            rows = table.find_all('tr')
            print(f"\n--- Table {table_idx} has {len(rows)} rows ---")

            # Show first few rows of each table
            for row_idx, row in enumerate(rows[:3]):  # First 3 rows only
                cells = row.find_all(['td', 'th'])
                cell_texts = [cell.get_text(strip=True) for cell in cells]
                print(f"  Row {row_idx}: {cell_texts}")

                # Check if this looks like a holdings table
                if any('CUSIP' in text.upper() for text in cell_texts):
                    print(f"  ✅ Table {table_idx} has CUSIP header!")
                if any('NAME OF ISSUER' in text.upper() for text in cell_texts):
                    print(f"  ✅ Table {table_idx} has NAME OF ISSUER header!")

        # IMPROVED TABLE DETECTION: Look for tables with proper headers, regardless of row count
        target_table = None
        for table in all_tables:
            rows = table.find_all('tr')

            # Check if it has typical 13F headers in first few rows
            for row in rows[:5]:  # Check first 5 rows for headers
                cells = row.find_all(['td', 'th'])
                cell_texts = [cell.get_text(strip=True).upper() for cell in cells]

                # Look for the characteristic 13F headers
                has_cusip = any('CUSIP' in text for text in cell_texts)
                has_issuer = any('NAME OF ISSUER' in text or 'ISSUER' in text for text in cell_texts)
                has_value = any('VALUE' in text or 'DOLLAR' in text for text in cell_texts)

                if has_cusip and has_issuer and has_value:
                    target_table = table
                    print(f"✅ Found valid holdings table with {len(rows)} rows")
                    break
            if target_table:
                break

        if not target_table:
            print("❌ Could not find a table with proper 13F headers")
            # Fallback: use any table that has CUSIP and ISSUER headers
            for table in all_tables:
                rows = table.find_all('tr')
                for row in rows[:3]:
                    cells = row.find_all(['td', 'th'])
                    cell_texts = [cell.get_text(strip=True).upper() for cell in cells]
                    has_cusip = any('CUSIP' in text for text in cell_texts)
                    has_issuer = any('NAME OF ISSUER' in text or 'ISSUER' in text for text in cell_texts)
                    if has_cusip and has_issuer:
                        target_table = table
                        print(f"⚠️  Using fallback table with {len(rows)} rows")
                        break
                if target_table:
                    break

        if not target_table:
            print("❌ Could not find any valid holdings table")
            return []

        # Parse the target table
        return parse_table_data(target_table)

    except Exception as e:
        print(f"❌ Error parsing HTML: {e}")
        return []

### Find the XML file URL from the documents page

In [9]:
def get_xml_file_url(documents_url, accession):
    """
    Find the XML file URL from the documents page

    Args:
        documents_url: URL to the filing documents page
        accession: Accession number

    Returns:
        URL to the primary XML file (form13fInfoTable.xml)
    """
    try:
        response = requests.get(documents_url, headers=HEADERS)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        # Look for the information table XML file
        table = soup.find('table', {'class': 'tableFile'})

        if not table:
            print("❌ No documents table found")
            return None

        rows = table.find_all('tr')[1:]  # Skip header

        # Look for XML files with various naming patterns
        xml_patterns = [
            'infotable.xml',
            'informationtable.xml',
            'form13f',
            '.xml'
        ]

        found_files = []

        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 3:
                filename = cols[2].text.strip().lower()

                # Check if this is an XML file we want
                for pattern in xml_patterns:
                    if pattern in filename and filename.endswith('.xml'):
                        link = cols[2].find('a')
                        if link:
                            xml_url = 'https://www.sec.gov' + link['href']
                            found_files.append({
                                'filename': filename,
                                'url': xml_url,
                                'priority': xml_patterns.index(pattern)
                            })
                            print(f"   Found XML file: {cols[2].text.strip()}")
                        break

        if not found_files:
            print("❌ No XML information table found")
            print("   Available files:")
            for row in rows[:10]:  # Show first 10 files
                cols = row.find_all('td')
                if len(cols) >= 3:
                    print(f"   - {cols[2].text.strip()}")
            return None

        # Return the highest priority XML file
        found_files.sort(key=lambda x: x['priority'])
        selected = found_files[0]
        print(f"✅ Selected XML file: {selected['filename']}")
        return selected['url']

    except Exception as e:
        print(f"❌ Error finding XML file: {e}")
        return None

### Parse the 13F XML file and extract holdings data

In [10]:
def parse_13f_html(html_url):
    """
    Parse the 13F HTML table and extract holdings
    """
    print(f"Downloading HTML table...")

    try:
        response = requests.get(html_url, headers=HEADERS)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        # Find ALL tables and examine each one
        all_tables = soup.find_all('table')
        print(f"Found {len(all_tables)} total tables in HTML")

        # IMPROVED TABLE DETECTION: Look for tables with proper headers
        target_table = None
        for table_idx, table in enumerate(all_tables):
            rows = table.find_all('tr')

            # Check if it has typical 13F headers
            for row in rows:
                cells = row.find_all(['td', 'th'])
                cell_texts = [cell.get_text(strip=True).upper() for cell in cells]

                # Look for the characteristic 13F headers
                has_cusip = any('CUSIP' in text for text in cell_texts)
                has_issuer = any('NAME OF ISSUER' in text or 'ISSUER' in text for text in cell_texts)
                has_value = any('VALUE' in text or 'DOLLAR' in text or '(TO THE NEAREST' in text for text in cell_texts)

                if has_cusip and has_issuer:
                    target_table = table
                    print(f"✅ Found valid holdings table at index {table_idx} with {len(rows)} rows")
                    break
            if target_table:
                break

        if not target_table:
            print("❌ Could not find a table with proper 13F headers")
            return []

        # Parse the target table
        return parse_table_data(target_table)

    except Exception as e:
        print(f"❌ Error parsing HTML: {e}")
        return []

In [11]:
def parse_table_data(table):
    """Parse the actual table data once we've found the right table"""
    rows = table.find_all('tr')
    print(f"Processing {len(rows)} rows in selected table")

    # Find header row - IMPROVED DETECTION
    header_row_idx = None
    column_headers = []

    for idx, row in enumerate(rows):
        cells = row.find_all(['th', 'td'])
        header_texts = [cell.get_text(strip=True).upper() for cell in cells]

        # Look for characteristic 13F headers - MORE FLEXIBLE
        has_cusip = any('CUSIP' in h for h in header_texts)
        has_issuer = any('NAME OF ISSUER' in h or 'ISSUER' in h for h in header_texts)
        has_value = any('VALUE' in h or 'DOLLAR' in h or '(TO THE NEAREST' in h for h in header_texts)

        # If we find all three key headers, this is definitely our header row
        if has_cusip and has_issuer and has_value:
            header_row_idx = idx
            column_headers = header_texts
            print(f"✅ Found header row at index {idx} (has CUSIP, ISSUER, VALUE)")
            print(f"   Headers: {column_headers}")
            break

        # Fallback: if we find CUSIP and ISSUER, it's probably our header row
        elif has_cusip and has_issuer:
            header_row_idx = idx
            column_headers = header_texts
            print(f"✅ Found header row at index {idx} (has CUSIP, ISSUER)")
            print(f"   Headers: {column_headers}")
            break

    # If still no header found, try looking for any row with CUSIP
    if header_row_idx is None:
        print("❌ Could not find header row with CUSIP and ISSUER columns")
        print("   Trying alternative header detection...")

        for idx, row in enumerate(rows):
            cells = row.find_all(['th', 'td'])
            if len(cells) > 3:
                cell_texts = [cell.get_text(strip=True) for cell in cells]

                # Look for any row that might contain CUSIP values (9-character alphanumeric)
                for text in cell_texts:
                    if len(text) == 9 and text.isalnum():

                        if idx > 0:
                            header_row_idx = idx - 1
                            prev_cells = rows[header_row_idx].find_all(['th', 'td'])
                            column_headers = [cell.get_text(strip=True).upper() for cell in prev_cells]
                            print(f"✅ Found header row at index {header_row_idx} (using CUSIP data detection)")
                            print(f"   Headers: {column_headers}")
                            break
                if header_row_idx is not None:
                    break

    if header_row_idx is None:
        print("❌ Could not find any valid header row")
        return []

    # Find column indices
    name_idx = next((i for i, h in enumerate(column_headers) if 'NAME' in h and 'ISSUER' in h), 0)
    cusip_idx = next((i for i, h in enumerate(column_headers) if 'CUSIP' in h), -1)
    value_idx = next((i for i, h in enumerate(column_headers) if 'VALUE' in h or 'DOLLAR' in h or '(TO THE NEAREST' in h), -1)
    shares_idx = next((i for i, h in enumerate(column_headers) if 'SHRS' in h or 'PRN AMT' in h or 'SH/' in h or 'PRN' in h), -1)

    # Look for columns that mention PUT, CALL, or PUT/CALL
    put_call_idx = -1
    for i, header in enumerate(column_headers):
        header_clean = header.replace('/', ' ').replace('&', ' ')  # Handle "PUT/CALL" or "PUT & CALL"
        if 'PUT' in header_clean and 'CALL' in header_clean:
            put_call_idx = i
            print(f"   ✅ Found PUT/CALL column at index {i}")
            break
        elif 'PUT' in header_clean:
            put_call_idx = i
            print(f"   ✅ Found PUT column at index {i}")
            break
        elif 'CALL' in header_clean:
            put_call_idx = i
            print(f"   ✅ Found CALL column at index {i}")
            break

    # If no dedicated PUT/CALL column, check if it's combined with shares
    if put_call_idx == -1:
        print(f"   ℹ️  No dedicated PUT/CALL column found")
        # Check if shares column header mentions put/call
        if shares_idx != -1 and shares_idx < len(column_headers):
            shares_header = column_headers[shares_idx]
            if 'PUT' in shares_header or 'CALL' in shares_header:
                print(f"   ℹ️  PUT/CALL info may be in shares column")
                put_call_idx = shares_idx

    title_class_idx = next((i for i, h in enumerate(column_headers) if 'TITLE' in h and 'CLASS' in h), -1)

    print(f"   Column indices - Name: {name_idx}, CUSIP: {cusip_idx}, Value: {value_idx}, Shares: {shares_idx}")
    print(f"   Put/Call column index: {put_call_idx}, Title of Class index: {title_class_idx}")

    if cusip_idx == -1:
        print("❌ Could not find CUSIP column")
        return []

    # Parse data rows
    data_rows = rows[header_row_idx + 1:]
    holdings = []

    for row_idx, row in enumerate(data_rows):
        cells = row.find_all(['td', 'th'])

        if len(cells) <= max(name_idx, cusip_idx, value_idx, shares_idx, put_call_idx):
            continue

        try:
            company_name = cells[name_idx].get_text(strip=True) if name_idx < len(cells) else ""
            cusip = cells[cusip_idx].get_text(strip=True) if cusip_idx < len(cells) else ""

            # Skip empty rows or non-holdings rows
            if not company_name or not cusip or len(cusip) != 9 or 'OMB' in company_name.upper():
                continue

            # Parse value (in thousands of dollars)
            value_text = cells[value_idx].get_text(strip=True) if value_idx != -1 and value_idx < len(cells) else "0"
            value = parse_number(value_text)

            # Parse shares
            shares_text = cells[shares_idx].get_text(strip=True) if shares_idx != -1 and shares_idx < len(cells) else "0"
            shares = parse_number(shares_text)

            # EXTRACT Put/Call information - IMPROVED LOGIC
            put_call_value = ""
            if put_call_idx != -1 and put_call_idx < len(cells):
                put_call_text = cells[put_call_idx].get_text(strip=True).upper()

                # Check for explicit "PUT" or "CALL" values
                if put_call_text in ['PUT', 'CALL']:
                    put_call_value = put_call_text
                # Check if it's combined with shares (like "5062500 SH PUT")
                elif 'PUT' in put_call_text or 'CALL' in put_call_text:
                    # Split and look for PUT/CALL keywords
                    parts = put_call_text.split()
                    for part in parts:
                        if part in ['PUT', 'CALL']:
                            put_call_value = part
                            break

            # Extract Title of Class (for reference only)
            title_class_value = ""
            if title_class_idx != -1 and title_class_idx < len(cells):
                title_class_value = cells[title_class_idx].get_text(strip=True).upper()

            # OPTION DETECTION - ONLY USING PUT/CALL VALUE
            is_put = put_call_value == 'PUT'
            is_call = put_call_value == 'CALL'
            is_option = is_put or is_call

            holding = {
                'company_name': company_name,
                'cusip': cusip,
                'value': value,
                'shares': shares,
                'share_type': 'SH',
                'put_call': put_call_value,
                'title_of_class': title_class_value,
                'is_option': is_option,
                'option_type': 'PUT' if is_put else 'CALL' if is_call else None
            }

            holdings.append(holding)

            # Debug first few rows AND any options found
            if row_idx < 3 or is_option:
                print(f"   DEBUG Row {row_idx}: Name='{company_name}', Put/Call='{put_call_value}', IsOption={is_option}")

        except Exception as e:
            print(f"   ❌ Error parsing row {row_idx}: {e}")
            continue

    print(f"✅ Successfully parsed {len(holdings)} valid holdings")

    # Count options for debugging
    options_count = sum(1 for h in holdings if h['is_option'])
    print(f"📊 Found {options_count} option positions out of {len(holdings)} total holdings")

    # Show examples of options found
    if options_count > 0:
        print(f"🔍 Options found:")
        option_holdings = [h for h in holdings if h['is_option']]
        for i, h in enumerate(option_holdings):
            print(f"   {i+1}. {h['company_name']} - {h['option_type']} - Shares: {h['shares']:,} - Value: ${h['value']:,}")

    # Show sample with actual values
    if holdings:
        total_value = sum(h['value'] for h in holdings)
        print(f"💰 Total portfolio value: ${total_value:,.0f}")
        print("\nFirst 5 holdings:")
        for i, h in enumerate(holdings[:5]):
            option_indicator = " [OPTION]" if h['is_option'] else ""
            print(f"   {i+1}. {h['company_name']}{option_indicator} - Shares: {h['shares']:,} - Value: ${h['value']:,.0f}")

    return holdings

In [12]:
def parse_number(text):
    """Parse numeric values from text - handle commas and formatting"""
    try:
        clean_text = ''.join(c for c in text if c.isdigit() or c == ',')
        clean_text = clean_text.replace(',', '')
        return int(clean_text) if clean_text else 0
    except Exception as e:
        print(f"   ❌ Number parsing error for '{text}': {e}")
        return 0

###  Main Extraction

In [13]:
def extract_13f_data(cik, period_date, save_to_file=True, force_refresh=False):
    """
    Complete pipeline: Extract 13F holdings data from HTML table

    Args:
        cik: Company CIK number
        period_date: Period end date in format 'YYYY-MM-DD'
        save_to_file: Whether to save to Google Drive
        force_refresh: Whether to re-extract even if file exists  # ADD THIS
    """
    # Check if already saved (unless force_refresh is True)
    if save_to_file and not force_refresh:
        filename = f"{cik}_{period_date.replace('-', '')}_holdings.pkl"
        filepath = data_dir / filename

        if filepath.exists():
            print(f"\n{'='*80}")
            print(f"LOADING FROM SAVED FILE")
            print(f"{'='*80}")
            df = pd.read_pickle(filepath)
            print(f"✅ Loaded {len(df)} holdings")
            print(f"   Total value: ${df['value'].sum():,.0f}")
            print(f"{'='*80}\n")
            return df

    print(f"\n{'='*80}")
    print(f"EXTRACTING 13F DATA")
    print(f"CIK: {cik}")
    print(f"Period: {period_date}")
    print(f"{'='*80}\n")

    # Step 1: Find the filing
    accession, documents_url = get_filing_url(cik, period_date)

    if not accession or not documents_url:
        return pd.DataFrame()

    time.sleep(0.5)

    # Step 2: Find the HTML table
    html_url = get_html_table_url(documents_url)

    if not html_url:
        return pd.DataFrame()

    time.sleep(0.5)

    # Step 3: Parse HTML and extract holdings
    holdings = parse_13f_html(html_url)

    if not holdings:
        return pd.DataFrame()
    df = pd.DataFrame(holdings)

    # Add metadata
    df['cik'] = str(cik).zfill(10)
    df['period_date'] = period_date
    df['accession_number'] = accession
    df['extraction_date'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # Save to file
    if save_to_file:
        filename = f"{cik}_{period_date.replace('-', '')}_holdings.pkl"
        filepath = data_dir / filename
        df.to_pickle(filepath)
        print(f"✅ Saved to: {filepath}")

        csv_filename = f"{cik}_{period_date.replace('-', '')}_holdings.csv"
        csv_filepath = data_dir / csv_filename
        df.to_csv(csv_filepath, index=False)
        print(f"✅ Saved CSV to: {csv_filepath}")

    print(f"\n{'='*80}")
    print(f"EXTRACTION COMPLETE")
    print(f"Total holdings: {len(df)}")
    print(f"Total value: ${df['value'].sum():,.0f}")
    print(f"{'='*80}\n")

    return df

### **COMPONENT 2: DATA PROCESSING**

In [14]:
def clean_holdings(df):
    """
    Clean and validate holdings data
    Removes Put and Call options

    Args:
        df: DataFrame with holdings

    Returns:
        Cleaned DataFrame
    """
    print(f"   Cleaning holdings data...")
    print(f"   Initial records: {len(df)}")

    # Make a copy to avoid modifying original
    df_clean = df.copy()

    # 1. Remove Put and Call options
    initial_count = len(df_clean)

    if 'is_option' in df_clean.columns:
        # Remove rows that are marked as options
        df_clean = df_clean[~df_clean['is_option']]
        removed = initial_count - len(df_clean)
        if removed > 0:
            print(f"   ✅ Removed {removed} Put/Call options")
    else:
        print(f"   ℹ️  No 'is_option' column found - checking for Put/Call columns")

        # Fallback: check for Put/Call in any column
        option_mask = False
        for col in df_clean.columns:
            if df_clean[col].dtype == 'object':  # String columns
                option_mask = option_mask | df_clean[col].str.upper().isin(['PUT', 'CALL']).fillna(False)

        if option_mask.any():
            df_clean = df_clean[~option_mask]
            removed = initial_count - len(df_clean)
            if removed > 0:
                print(f"   ✅ Removed {removed} Put/Call options (fallback method)")

    # 2. Validate CUSIP format (must be 9 characters)
    if 'cusip' in df_clean.columns:
        initial_count = len(df_clean)
        df_clean = df_clean[df_clean['cusip'].str.len() == 9]
        removed = initial_count - len(df_clean)
        if removed > 0:
            print(f"   ✅ Removed {removed} invalid CUSIPs")

    # 3. Remove duplicates (keep first)
    if 'cusip' in df_clean.columns:
        initial_count = len(df_clean)
        df_clean = df_clean.drop_duplicates(subset=['cusip'], keep='first')
        removed = initial_count - len(df_clean)
        if removed > 0:
            print(f"   ✅ Removed {removed} duplicate CUSIPs")

    # 4. Ensure numeric columns are correct type
    if 'shares' in df_clean.columns:
        df_clean['shares'] = pd.to_numeric(df_clean['shares'], errors='coerce').fillna(0).astype(int)
    if 'value' in df_clean.columns:
        df_clean['value'] = pd.to_numeric(df_clean['value'], errors='coerce').fillna(0).astype(int)

    # 5. Remove zero or negative values
    initial_count = len(df_clean)
    if 'shares' in df_clean.columns and 'value' in df_clean.columns:
        df_clean = df_clean[(df_clean['shares'] > 0) | (df_clean['value'] > 0)]
        removed = initial_count - len(df_clean)
        if removed > 0:
            print(f"   ✅ Removed {removed} records with zero/negative values")

    print(f"   Final records: {len(df_clean)}")
    print(f"   ✅ Cleaning complete\n")

    return df_clean

In [15]:
# ========== STEP 4: Comparison Algorithm ==========

def compare_holdings(old_df, new_df):
    """
    Compare two quarters of holdings and categorize changes

    Args:
        old_df: DataFrame for old period (e.g., Q1)
        new_df: DataFrame for new period (e.g., Q2)

    Returns:
        DataFrame with comparison results and classifications
    """
    print(f"\n{'='*80}")
    print(f"COMPARING HOLDINGS")
    print(f"{'='*80}")

    # Clean both DataFrames
    old_clean = clean_holdings(old_df)
    new_clean = clean_holdings(new_df)

    print(f"Old period: {len(old_clean)} positions")
    print(f"New period: {len(new_clean)} positions")

    # Create dictionaries keyed by CUSIP for fast lookup
    old_dict = old_clean.set_index('cusip')[['company_name', 'shares', 'value']].to_dict('index')
    new_dict = new_clean.set_index('cusip')[['company_name', 'shares', 'value']].to_dict('index')

    # Get all unique CUSIPs across both periods
    all_cusips = set(old_dict.keys()) | set(new_dict.keys())
    print(f"Total unique securities: {len(all_cusips)}\n")

    # Analyze each CUSIP
    comparison_results = []

    for cusip in all_cusips:
        # Get old position (default to 0 if not present)
        old_pos = old_dict.get(cusip, {'company_name': '', 'shares': 0, 'value': 0})
        old_shares = old_pos['shares']
        old_value = old_pos['value']
        old_name = old_pos['company_name']

        # Get new position (default to 0 if not present)
        new_pos = new_dict.get(cusip, {'company_name': '', 'shares': 0, 'value': 0})
        new_shares = new_pos['shares']
        new_value = new_pos['value']
        new_name = new_pos['company_name']

        # Use new name if available, otherwise old name
        company_name = new_name if new_name else old_name

        # Calculate changes
        shares_change = new_shares - old_shares
        value_change = new_value - old_value

        # Calculate percentage changes (avoid division by zero)
        shares_change_pct = (shares_change / old_shares * 100) if old_shares > 0 else None
        value_change_pct = (value_change / old_value * 100) if old_value > 0 else None

        # Classify the change based on SHARES (more reliable than value)
        if old_shares == 0 and new_shares > 0:
            change_type = 'NEW'
        elif old_shares > 0 and new_shares == 0:
            change_type = 'CLOSED'
        elif new_shares > old_shares:
            change_type = 'INCREASED'
        elif new_shares < old_shares:
            change_type = 'DECREASED'
        else:
            change_type = 'UNCHANGED'

        # Store result
        result = {
            'cusip': cusip,
            'company_name': company_name,
            'old_shares': old_shares,
            'new_shares': new_shares,
            'shares_change': shares_change,
            'shares_change_pct': shares_change_pct,
            'old_value': old_value,
            'new_value': new_value,
            'value_change': value_change,
            'value_change_pct': value_change_pct,
            'change_type': change_type
        }

        comparison_results.append(result)

    # Convert to DataFrame
    comparison_df = pd.DataFrame(comparison_results)

    # Print summary
    print(f"{'='*80}")
    print(f"COMPARISON SUMMARY")
    print(f"{'='*80}")
    change_counts = comparison_df['change_type'].value_counts()
    for change_type, count in change_counts.items():
        print(f"{change_type:12} : {count:5} positions")
    print(f"{'='*80}\n")

    return comparison_df

In [16]:
def calculate_portfolio_metrics(old_df, new_df, comparison_df):
    """
    Calculate key portfolio metrics and changes

    Args:
        old_df: DataFrame for old period
        new_df: DataFrame for new period
        comparison_df: DataFrame from compare_holdings()

    Returns:
        Dictionary with all metrics
    """
    print(f"\n{'='*80}")
    print(f"CALCULATING PORTFOLIO METRICS")
    print(f"{'='*80}\n")

    metrics = {}

    # 1. Total portfolio values
    old_total = old_df['value'].sum()
    new_total = new_df['value'].sum()
    total_change = new_total - old_total
    total_change_pct = (total_change / old_total * 100) if old_total > 0 else 0

    metrics['old_total_value'] = old_total
    metrics['new_total_value'] = new_total
    metrics['total_value_change'] = total_change
    metrics['total_value_change_pct'] = total_change_pct

    print(f"Portfolio Value:")
    print(f"  Old period : ${old_total:>15,.0f}")
    print(f"  New period : ${new_total:>15,.0f}")
    print(f"  Change     : ${total_change:>15,.0f} ({total_change_pct:+.2f}%)\n")

    # 2. Number of positions
    old_positions = len(old_df)
    new_positions = len(new_df)

    metrics['old_positions'] = old_positions
    metrics['new_positions'] = new_positions
    metrics['positions_change'] = new_positions - old_positions

    print(f"Number of Positions:")
    print(f"  Old period : {old_positions:>6}")
    print(f"  New period : {new_positions:>6}")
    print(f"  Change     : {new_positions - old_positions:>6}\n")

    # 3. Portfolio concentration (Top 10 holdings)
    old_top10_value = old_df.nlargest(10, 'value')['value'].sum()
    new_top10_value = new_df.nlargest(10, 'value')['value'].sum()

    old_concentration = (old_top10_value / old_total * 100) if old_total > 0 else 0
    new_concentration = (new_top10_value / new_total * 100) if new_total > 0 else 0

    metrics['old_top10_concentration'] = old_concentration
    metrics['new_top10_concentration'] = new_concentration
    metrics['concentration_change'] = new_concentration - old_concentration

    print(f"Top 10 Concentration:")
    print(f"  Old period : {old_concentration:.2f}%")
    print(f"  New period : {new_concentration:.2f}%")
    print(f"  Change     : {new_concentration - old_concentration:+.2f}%\n")

    # 4. Turnover rate
    new_count = len(comparison_df[comparison_df['change_type'] == 'NEW'])
    closed_count = len(comparison_df[comparison_df['change_type'] == 'CLOSED'])
    avg_positions = (old_positions + new_positions) / 2

    turnover_rate = ((new_count + closed_count) / avg_positions * 100) if avg_positions > 0 else 0

    metrics['new_positions_count'] = new_count
    metrics['closed_positions_count'] = closed_count
    metrics['turnover_rate'] = turnover_rate

    print(f"Portfolio Turnover:")
    print(f"  New positions    : {new_count:>6}")
    print(f"  Closed positions : {closed_count:>6}")
    print(f"  Turnover rate    : {turnover_rate:.2f}%\n")

    # 5. Largest position changes
    # Largest increases by absolute value change
    largest_increases = comparison_df[comparison_df['change_type'] == 'INCREASED'].nlargest(10, 'value_change')

    # Largest decreases by absolute value change
    largest_decreases = comparison_df[comparison_df['change_type'] == 'DECREASED'].nsmallest(10, 'value_change')

    metrics['largest_increases'] = largest_increases
    metrics['largest_decreases'] = largest_decreases

    print(f"Largest Increases (by $ value):")
    for idx, row in largest_increases.head(5).iterrows():
        print(f"  {row['company_name'][:40]:40} : ${row['value_change']:>15,.0f} ({row['value_change_pct']:>6.1f}%)")

    print(f"\nLargest Decreases (by $ value):")
    for idx, row in largest_decreases.head(5).iterrows():
        print(f"  {row['company_name'][:40]:40} : ${row['value_change']:>15,.0f} ({row['value_change_pct']:>6.1f}%)")

    print(f"\n{'='*80}\n")

    return metrics


In [17]:
def generate_summary_report(old_df, new_df, comparison_df, metrics, cik, old_period, new_period):
    """
    Generate a comprehensive summary report

    Args:
        old_df: Old holdings DataFrame
        new_df: New holdings DataFrame
        comparison_df: Comparison DataFrame
        metrics: Metrics dictionary
        cik: Institution CIK
        old_period: Old period date
        new_period: New period date

    Returns:
        Formatted string report
    """
    report = []
    report.append("=" * 80)
    report.append(f"13F HOLDINGS COMPARISON REPORT")
    report.append("=" * 80)
    report.append(f"Institution CIK: {cik}")
    report.append(f"Comparison: {old_period} vs {new_period}")
    report.append("=" * 80)
    report.append("")

    # Portfolio overview
    report.append("PORTFOLIO OVERVIEW:")
    report.append(f"  Total Value Change: ${metrics['total_value_change']:,.0f} ({metrics['total_value_change_pct']:+.2f}%)")
    report.append(f"  Position Count Change: {metrics['positions_change']:+d}")
    report.append(f"  Top 10 Concentration: {metrics['old_top10_concentration']:.1f}% → {metrics['new_top10_concentration']:.1f}%")
    report.append(f"  Portfolio Turnover: {metrics['turnover_rate']:.1f}%")
    report.append("")

    # Change breakdown
    report.append("CHANGE BREAKDOWN:")
    change_counts = comparison_df['change_type'].value_counts()
    for change_type in ['NEW', 'INCREASED', 'UNCHANGED', 'DECREASED', 'CLOSED']:
        count = change_counts.get(change_type, 0)
        report.append(f"  {change_type:12} : {count:5} positions")
    report.append("")

    # Top new positions
    new_positions = comparison_df[comparison_df['change_type'] == 'NEW'].nlargest(10, 'new_value')
    if len(new_positions) > 0:
        report.append("TOP 10 NEW POSITIONS:")
        for idx, row in new_positions.iterrows():
            report.append(f"  {row['company_name'][:45]:45} : ${row['new_value']:>15,.0f}")
        report.append("")

    # Top closed positions
    closed_positions = comparison_df[comparison_df['change_type'] == 'CLOSED'].nlargest(10, 'old_value')
    if len(closed_positions) > 0:
        report.append("TOP 10 CLOSED POSITIONS:")
        for idx, row in closed_positions.iterrows():
            report.append(f"  {row['company_name'][:45]:45} : ${row['old_value']:>15,.0f}")
        report.append("")

    # Largest increases
    if len(metrics['largest_increases']) > 0:
        report.append("TOP 10 INCREASED POSITIONS:")
        for idx, row in metrics['largest_increases'].iterrows():
            report.append(f"  {row['company_name'][:35]:35} : ${row['value_change']:>12,.0f} ({row['value_change_pct']:>6.1f}%)")
        report.append("")

    # Largest decreases
    if len(metrics['largest_decreases']) > 0:
        report.append("TOP 10 DECREASED POSITIONS:")
        for idx, row in metrics['largest_decreases'].iterrows():
            report.append(f"  {row['company_name'][:35]:35} : ${row['value_change']:>12,.0f} ({row['value_change_pct']:>6.1f}%)")
        report.append("")

    report.append("=" * 80)

    return "\n".join(report)

### **COMPONENT 3: Sector Mapping**

In [18]:
# ========== SIC CODE TO INDUSTRY SECTOR MAPPING ==========

def get_industry_sector_from_sic(sic_code):
    """
    Map SIC code to industry sector

    Args:
        sic_code: 4-digit SIC code (as string or int)

    Returns:
        Industry sector name
    """
    if pd.isna(sic_code) or sic_code == '':
        return 'Unknown'

    try:
        sic = int(sic_code)
    except (ValueError, TypeError):
        return 'Unknown'

    # SIC Code ranges mapped to industry sectors
    if 100 <= sic <= 999:
        return 'Agriculture, Forestry & Fishing'
    elif 1000 <= sic <= 1499:
        return 'Mining'
    elif 1500 <= sic <= 1799:
        return 'Construction'
    elif 2000 <= sic <= 3999:
        # Manufacturing - break down into subsectors
        if 2000 <= sic <= 2099:
            return 'Food & Beverage'
        elif 2100 <= sic <= 2199:
            return 'Tobacco'
        elif 2200 <= sic <= 2299:
            return 'Textiles & Apparel'
        elif 2300 <= sic <= 2399:
            return 'Textiles & Apparel'
        elif 2400 <= sic <= 2499:
            return 'Lumber & Wood Products'
        elif 2500 <= sic <= 2599:
            return 'Furniture & Fixtures'
        elif 2600 <= sic <= 2699:
            return 'Paper & Allied Products'
        elif 2700 <= sic <= 2799:
            return 'Printing & Publishing'
        elif 2800 <= sic <= 2899:
            return 'Chemicals & Pharmaceuticals'
        elif 2900 <= sic <= 2999:
            return 'Petroleum & Coal Products'
        elif 3000 <= sic <= 3099:
            return 'Rubber & Plastics'
        elif 3100 <= sic <= 3199:
            return 'Leather Products'
        elif 3200 <= sic <= 3299:
            return 'Stone, Clay & Glass'
        elif 3300 <= sic <= 3399:
            return 'Primary Metals'
        elif 3400 <= sic <= 3499:
            return 'Fabricated Metal Products'
        elif 3500 <= sic <= 3599:
            return 'Industrial Machinery & Equipment'
        elif 3600 <= sic <= 3699:
            return 'Electronic & Electric Equipment'
        elif 3700 <= sic <= 3799:
            return 'Transportation Equipment'
        elif 3800 <= sic <= 3899:
            return 'Instruments & Related Products'
        elif 3900 <= sic <= 3999:
            return 'Miscellaneous Manufacturing'
    elif 4000 <= sic <= 4999:
        # Transportation & Utilities
        if 4800 <= sic <= 4899:
            return 'Telecommunications'
        elif 4900 <= sic <= 4999:
            return 'Electric, Gas & Sanitary Services'
        else:
            return 'Transportation Services'
    elif 5000 <= sic <= 5199:
        return 'Wholesale Trade'
    elif 5200 <= sic <= 5999:
        return 'Retail Trade'
    elif 6000 <= sic <= 6799:
        # Finance, Insurance & Real Estate
        if 6000 <= sic <= 6299:
            return 'Banking & Financial Services'
        elif 6300 <= sic <= 6499:
            return 'Insurance'
        elif 6500 <= sic <= 6599:
            return 'Real Estate'
        elif 6700 <= sic <= 6799:
            return 'Investment & Asset Management'
    elif 7000 <= sic <= 8999:
        # Services
        if 7000 <= sic <= 7099:
            return 'Hotels & Lodging'
        elif 7200 <= sic <= 7299:
            return 'Personal Services'
        elif 7300 <= sic <= 7399:
            return 'Business Services'
        elif 7370 <= sic <= 7379:
            return 'Technology & Software'
        elif 7500 <= sic <= 7599:
            return 'Automotive Services'
        elif 7800 <= sic <= 7899:
            return 'Entertainment & Recreation'
        elif 8000 <= sic <= 8099:
            return 'Healthcare Services'
        elif 8200 <= sic <= 8299:
            return 'Education Services'
        elif 8300 <= sic <= 8399:
            return 'Social Services'
        elif 8600 <= sic <= 8699:
            return 'Membership Organizations'
        elif 8700 <= sic <= 8799:
            return 'Engineering & Management Services'
        else:
            return 'Services'
    elif 9000 <= sic <= 9999:
        return 'Public Administration'
    else:
        return 'Unknown'


In [19]:
# Global cache for SEC company data
_SEC_COMPANIES_CACHE = None

def load_sec_companies_data():
    """
    Load SEC company tickers data once and cache it globally
    Returns DataFrame with company info
    """
    global _SEC_COMPANIES_CACHE

    if _SEC_COMPANIES_CACHE is not None:
        return _SEC_COMPANIES_CACHE

    try:
        url = "https://www.sec.gov/files/company_tickers.json"
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()

        companies = response.json()
        companies_df = pd.DataFrame.from_dict(companies, orient='index')

        # Normalize company names for better matching
        companies_df['title_normalized'] = companies_df['title'].str.upper().str.strip()
        companies_df['ticker_normalized'] = companies_df['ticker'].str.upper().str.strip()

        _SEC_COMPANIES_CACHE = companies_df
        print(f"✅ Loaded {len(companies_df)} companies from SEC database")
        return companies_df

    except Exception as e:
        print(f"⚠️ Failed to load SEC companies data: {e}")
        return pd.DataFrame()

In [20]:
def get_sector_info_from_cusip(cusip, api_key=None, company_name=None):
    """
    Map CUSIP to INDUSTRY sector using SEC data and SIC codes

    Strategy:
    1. Match company name to SEC company tickers to get CIK
    2. Use CIK to fetch detailed company info including SIC code
    3. Map SIC code to industry sector

    Args:
        cusip: 9-character CUSIP
        api_key: Not used (kept for compatibility)
        company_name: Company name from 13F filing (required for matching)

    Returns:
        Dictionary with sector information
    """
    try:
        # Load SEC companies data (cached after first call)
        companies_df = load_sec_companies_data()

        if companies_df.empty:
            raise Exception("SEC companies data not available")

        # Initialize return values
        sic_code = None
        cik = None
        ticker = ''
        name = company_name or ''
        sic_description = ''

        if not company_name:
            raise Exception("Company name required for matching")

        # Normalize the company name for matching
        name_normalized = company_name.upper().strip()

        # Try exact match first
        exact_match = companies_df[companies_df['title_normalized'] == name_normalized]

        if not exact_match.empty:
            row = exact_match.iloc[0]
            cik = str(row['cik_str']).zfill(10)
            ticker = row.get('ticker', '')
            name = row.get('title', company_name)
        else:
            # Try partial match on first significant word
            # Remove common suffixes
            name_clean = name_normalized.replace(' INC', '').replace(' CORP', '').replace(' LTD', '')
            name_clean = name_clean.replace(' LLC', '').replace(' CO', '').replace(' THE', '').strip()

            if name_clean:
                # Get first significant word
                first_word = name_clean.split()[0] if name_clean.split() else ''

                if len(first_word) > 2:
                    partial_match = companies_df[
                        companies_df['title_normalized'].str.contains(first_word, case=False, na=False)
                    ]

                    if not partial_match.empty:
                        row = partial_match.iloc[0]
                        cik = str(row['cik_str']).zfill(10)
                        ticker = row.get('ticker', '')
                        name = row.get('title', company_name)

        # Fetch SIC code from submissions endpoint using CIK
        if cik:
            try:
                submissions_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
                sub_response = requests.get(submissions_url, headers=HEADERS)
                sub_response.raise_for_status()
                sub_data = sub_response.json()

                # Get SIC code and description
                sic_code = sub_data.get('sic')
                sic_description = sub_data.get('sicDescription', '')
                name = sub_data.get('name', name)  # Use official SEC name

                # Delay
                time.sleep(0.1)

            except Exception as e:
                print(f"   ⚠️ Could not fetch SIC for CIK {cik}: {e}")

        # Map SIC to industry sector
        industry_sector = get_industry_sector_from_sic(sic_code)

        return {
            'cusip': cusip,
            'ticker': ticker,
            'name': name,
            'exchange': '',
            'sector': industry_sector,
            'industry': sic_description,
            'sic': str(sic_code) if sic_code else '',
            'sic_description': sic_description
        }

    except Exception as e:
        print(f"   ⚠️ Could not map CUSIP {cusip} ({company_name}): {e}")
        return {
            'cusip': cusip,
            'ticker': '',
            'name': company_name or '',
            'exchange': '',
            'sector': 'Unknown',
            'industry': 'Unknown',
            'sic': '',
            'sic_description': ''
        }

In [21]:
def batch_map_cusips(holdings_df, cache_file=None, rate_limit_delay=0.2):
    """
    Map CUSIPs to sector information with caching
    Uses company names from holdings to improve matching accuracy

    Args:
        holdings_df: DataFrame with 'cusip' and 'company_name' columns
        cache_file: Path to cache file (pkl)
        rate_limit_delay: Delay between lookups in seconds

    Returns:
        DataFrame with sector mappings
    """
    print(f"\n{'='*80}")
    print(f"MAPPING CUSIPs TO SECTORS (SEC-only, no external APIs)")
    print(f"{'='*80}")

    # Get unique CUSIPs with their company names
    cusip_name_map = holdings_df[['cusip', 'company_name']].drop_duplicates('cusip')
    cusips_to_process = cusip_name_map['cusip'].tolist()

    print(f"Total CUSIPs to map: {len(cusips_to_process)}")

    # Load cache if exists
    cache = {}
    if cache_file and Path(cache_file).exists():
        cached_df = pd.read_pickle(cache_file)
        # Ensure cusip is a column, not index
        if 'cusip' not in cached_df.columns:
            cached_df = cached_df.reset_index()
            if 'index' in cached_df.columns:
                cached_df = cached_df.rename(columns={'index': 'cusip'})
        cache = cached_df.set_index('cusip').to_dict('index')
        print(f"✅ Loaded {len(cache)} cached mappings")

    # Identify CUSIPs that need mapping
    cusips_to_map = cusip_name_map[~cusip_name_map['cusip'].isin(cache.keys())]
    print(f"CUSIPs to fetch: {len(cusips_to_map)}")

    # Map new CUSIPs (with company names for better matching)
    for idx, (i, row) in enumerate(cusips_to_map.iterrows()):
        if (idx + 1) % 10 == 0:
            print(f"   Progress: {idx+1}/{len(cusips_to_map)}")

        cusip = row['cusip']
        company_name = row['company_name']

        # Pass company name to improve matching
        mapping = get_sector_info_from_cusip(cusip, company_name=company_name)
        cache[cusip] = mapping

        time.sleep(rate_limit_delay)

    # Build DataFrame from cache
    all_mappings = pd.DataFrame.from_records(
        [{'cusip': k, **v} for k, v in cache.items()]
    )

    # Save updated cache
    if cache_file:
        all_mappings.to_pickle(cache_file)
        print(f"✅ Saved {len(all_mappings)} mappings to cache")

    print(f"{'='*80}\n")
    return all_mappings


In [22]:
def enrich_holdings_with_sectors(holdings_df, sector_mappings_df):
    """
    Add sector information to holdings DataFrame

    Args:
        holdings_df: DataFrame with holdings (must have 'cusip' column)
        sector_mappings_df: DataFrame with sector mappings

    Returns:
        Enriched holdings DataFrame
    """
    print(f"Enriching holdings with sector data...")

    # Merge on CUSIP
    enriched = holdings_df.merge(
        sector_mappings_df[['cusip', 'ticker', 'sector', 'industry', 'sic_description']],
        on='cusip',
        how='left'
    )

    # Fill missing sectors with 'Unknown'
    enriched['sector'] = enriched['sector'].fillna('Unknown')
    enriched['industry'] = enriched['industry'].fillna('Unknown')

    # Report statistics
    unknown_count = (enriched['sector'] == 'Unknown').sum()
    known_count = len(enriched) - unknown_count

    print(f"   ✅ Enriched {len(enriched)} holdings")
    print(f"   Known sectors: {known_count}")
    print(f"   Unknown sectors: {unknown_count}\n")

    return enriched

In [23]:
# ========== SECTOR-LEVEL AGGREGATION FUNCTIONS ==========

def aggregate_by_sector(holdings_df):
    """
    Aggregate holdings by sector

    Args:
        holdings_df: Enriched holdings DataFrame with 'sector' column

    Returns:
        DataFrame with sector-level aggregations
    """
    if 'sector' not in holdings_df.columns:
        print("⚠️ No 'sector' column found. Run enrich_holdings_with_sectors first.")
        return pd.DataFrame()

    # Group by sector
    sector_agg = holdings_df.groupby('sector').agg({
        'value': 'sum',
        'shares': 'sum',
        'cusip': 'count'  # Number of positions
    }).reset_index()

    sector_agg.columns = ['sector', 'total_value', 'total_shares', 'num_positions']

    # Portfolio percentage
    total_portfolio_value = holdings_df['value'].sum()
    sector_agg['pct_of_portfolio'] = (sector_agg['total_value'] / total_portfolio_value * 100)

    # Sort by value
    sector_agg = sector_agg.sort_values('total_value', ascending=False)

    return sector_agg

In [24]:
def compare_sectors_across_periods(old_holdings_enriched, new_holdings_enriched):
    """
    Compare sector allocations between two periods

    Args:
        old_holdings_enriched: Old period holdings with sector data
        new_holdings_enriched: New period holdings with sector data

    Returns:
        DataFrame with sector-level comparison
    """
    print(f"\n{'='*80}")
    print(f"COMPARING SECTORS ACROSS PERIODS")
    print(f"{'='*80}\n")

    # Aggregate both periods by sector
    old_sectors = aggregate_by_sector(old_holdings_enriched)
    new_sectors = aggregate_by_sector(new_holdings_enriched)

    # Merge on sector
    comparison = pd.merge(
        old_sectors,
        new_sectors,
        on='sector',
        how='outer',
        suffixes=('_old', '_new')
    ).fillna(0)

    # Calculate changes
    comparison['value_change'] = comparison['total_value_new'] - comparison['total_value_old']
    comparison['value_change_pct'] = (
        (comparison['value_change'] / comparison['total_value_old'] * 100)
        .replace([float('inf'), -float('inf')], 0)
        .fillna(0)
    )

    comparison['pct_of_portfolio_change'] = (
        comparison['pct_of_portfolio_new'] - comparison['pct_of_portfolio_old']
    )

    comparison['position_change'] = (
        comparison['num_positions_new'] - comparison['num_positions_old']
    )

# Classify change type
    def classify_sector_change(row):
        if row['total_value_old'] == 0 and row['total_value_new'] > 0:
            return 'NEW'
        elif row['total_value_old'] > 0 and row['total_value_new'] == 0:
            return 'CLOSED'
        elif row['total_value_new'] > row['total_value_old']:
            return 'INCREASED'
        elif row['total_value_new'] < row['total_value_old']:
            return 'DECREASED'
        else:
            return 'UNCHANGED'

    comparison['change_type'] = comparison.apply(classify_sector_change, axis=1)

    # Sort by absolute value change
    comparison = comparison.sort_values('value_change', ascending=False, key=abs)

    print(f"SECTOR COMPARISON SUMMARY")
    print(f"{'-'*80}")
    change_counts = comparison['change_type'].value_counts()
    for change_type, count in change_counts.items():
        print(f"{change_type:12} : {count:3} sectors")

    print(f"\nTop 5 Increased Sectors (by $ value):")
    increased = comparison[comparison['change_type'] == 'INCREASED'].nlargest(5, 'value_change')
    for _, row in increased.iterrows():
        print(f"  {row['sector']:30} : ${row['value_change']:>15,.0f} ({row['value_change_pct']:>6.1f}%)")

    print(f"\nTop 5 Decreased Sectors (by $ value):")
    decreased = comparison[comparison['change_type'] == 'DECREASED'].nsmallest(5, 'value_change')
    for _, row in decreased.iterrows():
        print(f"  {row['sector']:30} : ${row['value_change']:>15,.0f} ({row['value_change_pct']:>6.1f}%)")

    print(f"\n{'='*80}\n")

    return comparison

In [25]:
def get_sector_details(holdings_enriched, sector_name):
    """
    Get detailed holdings for a specific sector

    Args:
        holdings_enriched: Enriched holdings DataFrame
        sector_name: Name of sector to analyze

    Returns:
        DataFrame with holdings in that sector
    """
    sector_holdings = holdings_enriched[holdings_enriched['sector'] == sector_name].copy()

    if len(sector_holdings) == 0:
        print(f"⚠️ No holdings found in sector: {sector_name}")
        return pd.DataFrame()

    sector_holdings = sector_holdings.sort_values('value', ascending=False)

    # Percentage within sector
    sector_total = sector_holdings['value'].sum()
    sector_holdings['pct_of_sector'] = (sector_holdings['value'] / sector_total * 100)

    return sector_holdings

In [26]:
def compare_sector_details(old_enriched, new_enriched, sector_name):
    """
    Compare position-level changes within a specific sector

    Args:
        old_enriched: Old period enriched holdings
        new_enriched: New period enriched holdings
        sector_name: Sector to analyze

    Returns:
        DataFrame with sector position comparison
    """
    print(f"\n{'='*80}")
    print(f"SECTOR DETAIL COMPARISON: {sector_name}")
    print(f"{'='*80}\n")

    # Get holdings for this sector in both periods
    old_sector = old_enriched[old_enriched['sector'] == sector_name][['cusip', 'company_name', 'value', 'shares']]
    new_sector = new_enriched[new_enriched['sector'] == sector_name][['cusip', 'company_name', 'value', 'shares']]

    # Merge
    comparison = pd.merge(
        old_sector,
        new_sector,
        on='cusip',
        how='outer',
        suffixes=('_old', '_new')
    ).fillna(0)

    # Use new name if available, otherwise old
    comparison['company_name'] = comparison['company_name_new'].where(
        comparison['company_name_new'] != '',
        comparison['company_name_old']
    )
    comparison = comparison.drop(['company_name_old', 'company_name_new'], axis=1)

    # Calculate changes
    comparison['value_change'] = comparison['value_new'] - comparison['value_old']
    comparison['shares_change'] = comparison['shares_new'] - comparison['shares_old']

    # Classify
    def classify_position(row):
        if row['value_old'] == 0 and row['value_new'] > 0:
            return 'NEW'
        elif row['value_old'] > 0 and row['value_new'] == 0:
            return 'CLOSED'
        elif row['value_new'] > row['value_old']:
            return 'INCREASED'
        elif row['value_new'] < row['value_old']:
            return 'DECREASED'
        else:
            return 'UNCHANGED'

    comparison['change_type'] = comparison.apply(classify_position, axis=1)

    # Sort by absolute value change
    comparison = comparison.sort_values('value_change', ascending=False, key=abs)

    print(f"Position Changes in {sector_name}:")
    change_counts = comparison['change_type'].value_counts()
    for change_type, count in change_counts.items():
        print(f"  {change_type:12} : {count:3} positions")

    print(f"\n{'='*80}\n")

    return comparison

In [27]:

# ========== COMPLETE WORKFLOW FUNCTION ==========
def create_sector_analysis(old_df, new_df, cache_file=None):
    """
    Complete workflow: enrich holdings with sectors and create sector comparison
    Uses only SEC data - no external APIs needed

    Args:
        old_df: Old period holdings DataFrame (with company_name column)
        new_df: New period holdings DataFrame (with company_name column)
        cache_file: Path to sector mapping cache

    Returns:
        Tuple of (old_enriched, new_enriched, sector_comparison)
    """
    print(f"\n{'='*80}")
    print(f"SECTOR ANALYSIS WORKFLOW")
    print(f"{'='*80}\n")

    # Step 1: Combine both periods to get all unique CUSIPs with names
    combined_df = pd.concat([
        old_df[['cusip', 'company_name']],
        new_df[['cusip', 'company_name']]
    ]).drop_duplicates('cusip')

    print(f"Step 1: Found {len(combined_df)} unique CUSIPs across both periods\n")

    # Step 2: Map CUSIPs to sectors using company names (with caching)
    sector_mappings = batch_map_cusips(combined_df, cache_file, rate_limit_delay=0.5)

    # Step 3: Enrich holdings with sector data
    print(f"Step 3: Enriching holdings with sector data")
    old_enriched = enrich_holdings_with_sectors(old_df, sector_mappings)
    new_enriched = enrich_holdings_with_sectors(new_df, sector_mappings)

    # Step 4: Compare sectors across periods
    print(f"Step 4: Comparing sectors across periods")
    sector_comparison = compare_sectors_across_periods(old_enriched, new_enriched)

    print(f"✅ Sector analysis complete!\n")
    print(f"{'='*80}\n")

    return old_enriched, new_enriched, sector_comparison


In [28]:
SECTOR_CACHE_FILE = data_dir / "sector_mappings_cache.pkl"

print("✅ Configuration set")

✅ API Configuration set


## **Main Workflow**

In [29]:
# ========== CELL: Complete Comparison Workflow ==========

# Configuration
CIK = "0001313360"
OLD_PERIOD = "2025-09-30"  # Q3 2025
NEW_PERIOD = "2025-12-31"  # Q4 2025

print(f"\n{'='*80}")
print(f"13F HOLDINGS ANALYSIS - 2025 Q3 vs Q4")
print(f"{'='*80}")
print(f"Institution: CIK {CIK}")
print(f"Periods: {OLD_PERIOD} vs {NEW_PERIOD}")
print(f"{'='*80}\n")

# Step 1: Extract or load holdings data WITH FORCE_REFRESH
print("STEP 1: Loading Holdings Data")
print("-" * 80)
old_df = extract_13f_data(CIK, OLD_PERIOD, force_refresh=True)
time.sleep(1)
new_df = extract_13f_data(CIK, NEW_PERIOD, force_refresh=True)

if old_df.empty or new_df.empty:
    print("\n❌ Failed to load holdings data")
    print("Note: Ensure filings are available for these periods")
else:
    print("\n✅ Holdings data loaded successfully\n")

    # Step 2: Compare holdings (includes Put/Call filtering)
    print("STEP 2: Comparing Holdings")
    print("-" * 80)
    comparison_df = compare_holdings(old_df, new_df)

    # Step 3: Calculate metrics
    print("STEP 3: Calculating Metrics")
    print("-" * 80)
    metrics = calculate_portfolio_metrics(old_df, new_df, comparison_df)

    # Step 4: Generate report
    print("STEP 4: Generating Report")
    print("-" * 80)
    report = generate_summary_report(
        old_df, new_df, comparison_df, metrics,
        CIK, OLD_PERIOD, NEW_PERIOD
    )
    print(report)


13F HOLDINGS ANALYSIS - 2025 Q3 vs Q4
Institution: CIK 0001313360
Periods: 2025-09-30 vs 2025-12-31

STEP 1: Loading Holdings Data
--------------------------------------------------------------------------------

EXTRACTING 13F DATA
CIK: 0001313360
Period: 2025-09-30

Searching for 13F filing: CIK 0001313360, Reporting Period 2025-09-30
  - Filing: 2026-01-09, Reporting: 2025-12-31
  - Filing: 2025-10-23, Reporting: 2025-09-30
✅ Found matching filing! Filed on 2025-10-23, reporting period 2025-09-30
   Searching for INFORMATION TABLE...
   ✅ Found: form13fInfoTable20251009.html
Found 4 total tables in HTML
✅ Found valid holdings table at index 3 with 3357 rows
Processing 3357 rows in selected table
✅ Found header row at index 2 (has CUSIP, ISSUER, VALUE)
   Headers: ['NAME OF ISSUER', 'TITLE OF CLASS', 'CUSIP', 'FIGI', '(TO THE NEAREST DOLLAR)', 'PRN AMT', 'PRN', 'CALL', 'DISCRETION', 'MANAGER', 'SOLE', 'SHARED', 'NONE']
   ✅ Found CALL column at index 7
   Column indices - Name: 0, C

In [30]:
# Delete old SEC-API cache
SECTOR_CACHE_FILE.unlink(missing_ok=True)

## **Workflow with Sectors**

In [31]:
# ========== CELL: Complete Analysis with Sector Mapping ==========

# Configuration
CIK = "0002096567"
OLD_PERIOD = "2025-06-30"  # Q3 2025
NEW_PERIOD = "2025-09-30"  # Q4 2025

print(f"\n{'='*80}")
print(f"13F HOLDINGS ANALYSIS WITH SECTOR MAPPING")
print(f"{'='*80}")
print(f"Institution: CIK {CIK}")
print(f"Periods: {OLD_PERIOD} vs {NEW_PERIOD}")
print(f"{'='*80}\n")

# Step 1: Extract holdings data
print("STEP 1: Loading Holdings Data")
print("-" * 80)
old_df = extract_13f_data(CIK, OLD_PERIOD)
time.sleep(1)
new_df = extract_13f_data(CIK, NEW_PERIOD)

if old_df.empty or new_df.empty:
    print("\n❌ Failed to load holdings data")
else:
    print("\n✅ Holdings data loaded successfully\n")

    # Step 2: Clean and compare holdings
    print("STEP 2: Comparing Holdings (Position Level)")
    print("-" * 80)
    comparison_df = compare_holdings(old_df, new_df)
    metrics = calculate_portfolio_metrics(old_df, new_df, comparison_df)

    # Step 3: SECTOR ANALYSIS
    print("STEP 3: Sector Analysis")
    print("-" * 80)
    old_enriched, new_enriched, sector_comparison = create_sector_analysis(
        old_df,
        new_df,
                cache_file=SECTOR_CACHE_FILE
    )

    # Save enriched data
    old_enriched.to_pickle(data_dir / f"{CIK}_{OLD_PERIOD.replace('-', '')}_enriched.pkl")
    new_enriched.to_pickle(data_dir / f"{CIK}_{NEW_PERIOD.replace('-', '')}_enriched.pkl")
    sector_comparison.to_pickle(data_dir / f"{CIK}_{OLD_PERIOD.replace('-', '')}_{NEW_PERIOD.replace('-', '')}_sector_comparison.pkl")

    print("✅ Saved enriched holdings and sector comparison to Drive\n")

    # Step 4: Display sector summary
    print("STEP 4: Sector Summary")
    print("-" * 80)
    print("\nSector Allocation (New Period):")
    print(new_enriched.groupby('sector')['value'].sum().sort_values(ascending=False).head(10))


13F HOLDINGS ANALYSIS WITH SECTOR MAPPING
Institution: CIK 0002096567
Periods: 2025-06-30 vs 2025-09-30

STEP 1: Loading Holdings Data
--------------------------------------------------------------------------------

LOADING FROM SAVED FILE
✅ Loaded 101 holdings
   Total value: $114,661,963


LOADING FROM SAVED FILE
✅ Loaded 95 holdings
   Total value: $115,929,252


✅ Holdings data loaded successfully

STEP 2: Comparing Holdings (Position Level)
--------------------------------------------------------------------------------

COMPARING HOLDINGS
   Cleaning holdings data...
   Initial records: 101
   Final records: 101
   ✅ Cleaning complete

   Cleaning holdings data...
   Initial records: 95
   Final records: 95
   ✅ Cleaning complete

Old period: 101 positions
New period: 95 positions
Total unique securities: 105

COMPARISON SUMMARY
DECREASED    :    34 positions
INCREASED    :    32 positions
UNCHANGED    :    25 positions
CLOSED       :    10 positions
NEW          :     4 positi

### **COMPONENT 4: LLM Integration**

In [32]:
# CELL: Install Google Gemini SDK
# !pip install -q google-generativeai

In [33]:
# LLM QUERY INTERFACE (GOOGLE GEMINI)

import google.generativeai as genai
import json

def setup_gemini_client(api_key):
    """Initialize Google Gemini client"""
    genai.configure(api_key=api_key)
    return genai.GenerativeModel('models/gemini-2.5-flash')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [34]:
# DATA PREPARATION FOR LLM

def prepare_context_for_llm(old_enriched, new_enriched, comparison_df, sector_comparison, metrics):
    """
    Prepare a concise context summary for the LLM

    Returns a dictionary with all the data the LLM needs
    """
    context = {
        "portfolio_summary": {
            "old_total_value": int(metrics['old_total_value']),
            "new_total_value": int(metrics['new_total_value']),
            "value_change": int(metrics['total_value_change']),
            "value_change_pct": round(metrics['total_value_change_pct'], 2),
            "old_positions": int(metrics['old_positions']),
            "new_positions": int(metrics['new_positions']),
            "positions_change": int(metrics['positions_change']),
            "turnover_rate": round(metrics['turnover_rate'], 2)
        },

        "position_changes": {
            "new": comparison_df[comparison_df['change_type'] == 'NEW'][['company_name', 'new_value', 'cusip']].to_dict('records'),
            "closed": comparison_df[comparison_df['change_type'] == 'CLOSED'][['company_name', 'old_value', 'cusip']].to_dict('records'),
            "increased": comparison_df[comparison_df['change_type'] == 'INCREASED'].nlargest(20, 'value_change')[['company_name', 'value_change', 'value_change_pct', 'old_value', 'new_value', 'cusip']].to_dict('records'),
            "decreased": comparison_df[comparison_df['change_type'] == 'DECREASED'].nsmallest(20, 'value_change')[['company_name', 'value_change', 'value_change_pct', 'old_value', 'new_value', 'cusip']].to_dict('records')
        },

        "sector_analysis": {
            "sectors": sector_comparison[['sector', 'total_value_old', 'total_value_new', 'value_change', 'value_change_pct', 'pct_of_portfolio_old', 'pct_of_portfolio_new', 'change_type']].to_dict('records')
        },

        "current_holdings": {
            "top_20_by_value": new_enriched.nlargest(20, 'value')[['company_name', 'sector', 'value', 'shares', 'ticker', 'cusip']].to_dict('records')
        }
    }

    return context

In [35]:
# Prompt Engineering

def create_system_prompt():
    """Create the system prompt for the LLM"""
    return """You are a financial analysis assistant specializing in 13F filings analysis.

You have access to detailed portfolio holdings data including:
- Position-level changes (new, closed, increased, decreased positions)
- Sector-level aggregations and changes
- Portfolio metrics (total value, concentration, turnover)
- Individual company holdings with sector classifications

When answering questions:
1. Be precise with numbers - always include dollar amounts and percentages when relevant
2. Format large numbers with commas (e.g., $1,234,567)
3. If asked about sectors, note that some positions may be classified as "Unknown" due to data availability
4. For company-specific questions, provide the company name, value, and change information
5. When comparing periods, clearly state "old period" vs "new period" or use the actual dates if provided
6. Be concise but comprehensive - don't leave out important details
7. If data is not available to answer a question, clearly state what information is missing

Remember: All values are in dollars (not thousands). Percentages should be formatted with 1-2 decimal places."""

In [36]:
#  QUERY PROCESSING

def query_portfolio(question, context, model):
    """
    Send a question to Gemini with the portfolio context

    Args:
        question: User's question
        context: Portfolio data context
        model: Gemini model instance

    Returns:
        Gemini's response
    """
    prompt = f"""{create_system_prompt()}

Portfolio Data Context:
{json.dumps(context, indent=2)}

User Question: {question}

Please answer the question using the portfolio data provided above. Be specific and cite actual numbers from the data."""

    try:
        response = model.generate_content(prompt)
        return response.text

    except Exception as e:
        return f"Error querying Gemini: {str(e)}"

In [38]:
# INTERACTIVE QUERY INTERFACE

class PortfolioAnalysisAgent:
    """
    Interactive agent for querying portfolio data
    """

    def __init__(self, old_enriched, new_enriched, comparison_df, sector_comparison, metrics, gemini_api_key):
        """Initialize the agent with portfolio data"""
        self.old_enriched = old_enriched
        self.new_enriched = new_enriched
        self.comparison_df = comparison_df
        self.sector_comparison = sector_comparison
        self.metrics = metrics

        # Setup Gemini client
        self.model = setup_gemini_client(gemini_api_key)
        self.context = prepare_context_for_llm(
            old_enriched, new_enriched, comparison_df, sector_comparison, metrics
        )

        print("✅ Portfolio Analysis Agent initialized (Google Gemini)")
        print(f"📊 Portfolio: ${metrics['new_total_value']:,.0f} ({metrics['new_positions']} positions)")
        print(f"📈 Change: ${metrics['total_value_change']:,.0f} ({metrics['total_value_change_pct']:+.2f}%)")

    def ask(self, question):
      print(f"\n{'='*80}")
      print(f"QUESTION: {question}")
      print(f"{'='*80}\n")

      answer = query_portfolio(question, self.context, self.model)

      print(f"\n{'='*80}\n")

      return answer

    def get_sector_details(self, sector_name):
        """Get detailed holdings for a specific sector"""
        holdings = self.new_enriched[self.new_enriched['sector'] == sector_name]

        if len(holdings) == 0:
            print(f"⚠️ No holdings found in sector: {sector_name}")
            return None

        print(f"\n{'='*80}")
        print(f"HOLDINGS IN {sector_name.upper()} SECTOR")
        print(f"{'='*80}\n")

        for idx, row in holdings.nlargest(20, 'value').iterrows():
            print(f"{row['company_name']:40} | ${row['value']:>15,.0f} | {row.get('ticker', 'N/A'):>6}")

        print(f"\nTotal: {len(holdings)} positions, ${holdings['value'].sum():,.0f}")
        print(f"{'='*80}\n")

        return holdings

In [42]:
# EXAMPLE QUERIES

def show_example_queries():
    """Display example questions users can ask"""
    examples = [
        "What sectors did the fund increase investment in?",
        "What were the top 5 new positions by value?",
        "How much did the fund's investment in the Equity sector grow?",
        "Which companies had the largest decreases?",
        "What is the current portfolio concentration?",
        "List all companies in the Healthcare sector",
        "How does the Equity sector compare to last quarter?",
        "What was the portfolio turnover rate?",
        "Show me the biggest position changes",
        "What percentage of the portfolio is in Unknown sectors?"
    ]

    print(f"\n{'='*80}")
    print("EXAMPLE QUESTIONS:")
    print(f"{'='*80}\n")

    for i, example in enumerate(examples, 1):
        print(f"{i:2}. {example}")

    print(f"\n{'='*80}\n")

In [43]:
#  Setup LLM Agent (Gemini)

GEMINI_API_KEY = "Your API key from https://aistudio.google.com/app/apikey"

# Create the analysis agent
agent = PortfolioAnalysisAgent(
    old_enriched=old_enriched,
    new_enriched=new_enriched,
    comparison_df=comparison_df,
    sector_comparison=sector_comparison,
    metrics=metrics,
    gemini_api_key=GEMINI_API_KEY
)

show_example_queries()

✅ Portfolio Analysis Agent initialized (Google Gemini)
📊 Portfolio: $115,929,252 (95 positions)
📈 Change: $1,267,289 (+1.11%)

EXAMPLE QUESTIONS:

 1. What sectors did the fund increase investment in?
 2. What were the top 5 new positions by value?
 3. How much did the fund's investment in the Equity sector grow?
 4. Which companies had the largest decreases?
 5. What is the current portfolio concentration?
 6. List all companies in the Healthcare sector
 7. How does the Equity sector compare to last quarter?
 8. What was the portfolio turnover rate?
 9. Show me the biggest position changes
10. What percentage of the portfolio is in Unknown sectors?




## **Interface**

In [41]:
agent.ask("Give me a succint one paragraph report of major changes in portfolio.")


QUESTION: Give me a succint one paragraph report of major changes in portfolio.





'The portfolio experienced a modest increase in total value, rising by $1,267,289 (1.11%) from $114,661,963 to $115,929,252, while reducing its number of positions by 6, from 101 to 95, with a turnover rate of 14.29%. Key activities included closing a substantial position in INVESCO QQQ TR valued at $5,460,084, and initiating a new significant holding in META PLATFORMS INC for $1,301,396. Among existing holdings, the largest increases were seen in ISHARES TR (CUSIP 464287226) by $550,638 (17.96%) and ISHARES TR (CUSIP 464288760) by $448,333 (20.41%), while UNITEDHEALTH GROUP INC saw the most significant decrease by $104,499 (-20.60%). Sector-wise, "Banking & Financial Services" increased its value by $2,308,840 (6.31%) to become the largest sector at $38,890,885 (33.55% of the portfolio), and "Business Services" surged by $966,227 (47.10%). Conversely, the "Unknown" sector experienced the largest value decrease of $3,056,164 (-8.95%), reducing its portfolio share to 26.82%.'